# Triagem Automática de Peças de Vestuário

Um marketplace de moda recebe fotos de peças enviadas por vendedores e precisa catalogá-las em uma das dez categorias do catálogo. Uma CNN faz a classificação e, quando a confiança da predição fica abaixo de um limiar, a peça é encaminhada para revisão humana em vez de ser catalogada automaticamente. O modelo assume apenas os casos em que está seguro.

O dataset utilizado é o **Fashion-MNIST**: 70.000 imagens em tons de
cinza de 28×28 pixels, divididas em 60.000 para treino e 10.000 para
teste, distribuídas igualmente entre 10 classes de vestuário.

Referências
- [TensorFlow: fashion_mnist](https://www.tensorflow.org/datasets/catalog/fashion_mnist)
- [Zalando Research: Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist)

## Setup

Sobre a alocação de GPU, o Keras 3 lê a variável `KERAS_BACKEND` no momento em que o pacote é
importado pela primeira vez.

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "torch")

## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import keras
import torch

from dataclasses import dataclass
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

## Carregar Dataset

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

## Sobre os Dados

### Pré-processamento

As imagens vêm como inteiros de 0 a 255 e são convertidas para `float32` na faixa `[0, 1]`.

A camada `Conv2D` espera um tensor com o eixo de canais explícito, então o formato passa de `(28, 28)` para `(28, 28, 1)`.

In [ ]:
CLASSES = (
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
)


x_train = np.expand_dims(x_train.astype("float32") / 255.0, axis=-1)
x_test = np.expand_dims(x_test.astype("float32") / 255.0, axis=-1)


print("Train images:", x_train.shape)
print("Test images:", x_test.shape)
print("Classes:", len(CLASSES))

### Visualização

Uma amostra do conjunto de treino e a contagem de exemplos por classe,
para confirmar que o dataset é balanceado.

In [ ]:
plt.figure(figsize=(8, 8))

for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.title(CLASSES[y_train[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
labels, counts = np.unique(y_train, return_counts=True)


df_counts = pd.DataFrame({
    "class": [CLASSES[label] for label in labels],
    "count": counts,
})


sns.barplot(df_counts, x="count", y="class", hue="class", legend=False)
plt.title("Exemplos por classe (treino)")
plt.tight_layout()
plt.show()

## Sobre o Modelo

### Linha de Base

Antes da CNN, uma rede densa simples estabelece o piso de desempenho: ela achata a imagem em um vetor de 784 pixels e, com isso, descarta toda a informação espacial.

O número que ela alcançar é a referência para medir o quanto as camadas convolucionais realmente agregam.

In [ ]:
baseline = keras.Sequential([
    keras.layers.Input(shape=(28, 28, 1)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
], name="baseline_mlp")

baseline.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

baseline.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=2,
)


_, baseline_acc = baseline.evaluate(x_test, y_test, verbose=0)
print(f"Baseline test accuracy: {baseline_acc:.2%}")

### Montar

A CNN segue a mesma estrutura vista em sala de aula, para o notebook do `cifar10`: dois blocos `Conv2D` + `MaxPooling2D` extraem padrões locais (bordas, texturas, contornos das peças) e um bloco denso faz a classificação final.

O `Dropout` antes da camada de saída reduz o sobreajuste.

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(28, 28, 1)),

    keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    keras.layers.MaxPooling2D(2, 2),

    keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    keras.layers.MaxPooling2D(2, 2),

    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),

    keras.layers.Dense(10, activation="softmax"),
], name="clothing_classifier")

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

### Treinamento

#### Alocar GPU

O treinamento roda na GPU do Apple Silicon (M4) por meio do backend **PyTorch** do Keras 3, que utiliza o dispositivo **MPS** (Metal Performance Shaders).

A alocação do dispositivo é automática: o backend PyTorch seleciona o MPS quando disponível.

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

print(f"Keras backend: {keras.backend.backend()}")
print(f"Training device: {device}")

#### Treinar

O `EarlyStopping` interrompe o treino quando a perda de validação para de melhorar e restaura os pesos da melhor época, evitando tanto o sobreajuste quanto épocas desperdiçadas.

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

logs = model.fit(
    x_train,
    y_train,
    epochs=20,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stopping],
    verbose=2,
)

### Avaliar

In [ ]:
lss, acc = model.evaluate(x_test, y_test, verbose=0)

print(f"lss: {lss:.4f}")
print(f"acc: {acc:.2%}")
print(f"Ganho sobre a linha de base: {(acc - baseline_acc):.2%}")

#### Curvas de Aprendizado

As curvas mostram se o modelo ainda estava aprendendo quando o treino parou e o quanto as métricas de treino e validação se afastaram.

In [ ]:
history = pd.DataFrame(logs.history)
history.index = history.index + 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

history[["loss", "val_loss"]].plot(ax=axes[0], marker="o")
axes[0].set_title("Perda")
axes[0].set_xlabel("Época")

history[["accuracy", "val_accuracy"]].plot(ax=axes[1], marker="o")
axes[1].set_title("Acurácia")
axes[1].set_xlabel("Época")

plt.tight_layout()
plt.show()

#### Matriz de Confusão

A matriz e o relatório por classe mostram quais categorias o modelo troca entre si (informação que orienta diretamente a etapa de triagem).

In [ ]:
probabilities = model.predict(x_test, verbose=0)
y_pred = np.argmax(probabilities, axis=1)

matrix = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASSES,
    yticklabels=CLASSES,
)
plt.title("Matriz de confusão (teste)")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=CLASSES,
    digits=3,
))

#### Erros do Modelo

Uma amostra das imagens classificadas incorretamente, com a confiança que o modelo atribuiu ao palpite errado.

Confianças altas em erros são justamente o caso que o limiar da triagem não consegue capturar.

In [ ]:
misclassified = np.flatnonzero(y_pred != y_test)

print("Erros:", len(misclassified))

plt.figure(figsize=(10, 8))

for position, index in enumerate(misclassified[:9]):
    confidence = probabilities[index].max()

    plt.subplot(3, 3, position + 1)
    plt.imshow(x_test[index].squeeze(), cmap="gray")
    plt.title(
        f"Real: {CLASSES[y_test[index]]}\n"
        f"Predito: {CLASSES[y_pred[index]]} ({confidence:.0%})",
        fontsize=9,
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

## Aplicação

A saída `softmax` da última camada é uma distribuição de probabilidade sobre as dez classes, e o maior valor dela funciona como um grau de confiança. A aplicação usa esse valor como critério de triagem: acima do limiar a peça é catalogada automaticamente; abaixo dele, entra na fila de revisão humana.

### Escolher o Limiar

In [ ]:
THRESHOLDS = (0.50, 0.70, 0.80, 0.90, 0.95, 0.99)


def screen(threshold: float) -> dict:
    """Measure the triage trade-off for a confidence threshold.

    :param threshold: Minimum confidence to catalog automatically.
    :type threshold: float
    :return: Threshold, automated share and accuracy on that share.
    :rtype: dict
    """
    confidences = probabilities.max(axis=1)
    accepted = confidences >= threshold
    hits = y_pred[accepted] == y_test[accepted]

    return {
        "threshold": threshold,
        "automated": accepted.mean(),
        "accuracy": hits.mean(),
        "for_review": (~accepted).sum(),
    }


df_screening = pd.DataFrame(
    [screen(threshold=threshold) for threshold in THRESHOLDS]
)

df_screening

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(
    df_screening["threshold"],
    df_screening["accuracy"],
    marker="o",
    label="Acurácia do que é catalogado",
)
ax.plot(
    df_screening["threshold"],
    df_screening["automated"],
    marker="o",
    label="Fração catalogada automaticamente",
)

ax.set_xlabel("Limiar de confiança")
ax.set_title("Cobertura x acurácia da triagem")
ax.legend()

plt.tight_layout()
plt.show()

### Salvar Modelo Treinado

In [ ]:
MODEL_PATH = "models/clothing_classifier.keras"

model.save(MODEL_PATH)

### Fazer Predição

In [ ]:
CONFIDENCE_THRESHOLD = 0.90

classifier = keras.saving.load_model(MODEL_PATH)


@dataclass(frozen=True)
class Screening:
    """Triage decision for a single clothing photo.

    :param label: Predicted catalog category.
    :type label: str
    :param confidence: Probability assigned to the predicted class.
    :type confidence: float
    :param needs_review: Whether a human must check the photo.
    :type needs_review: bool
    """

    label: str
    confidence: float
    needs_review: bool


def classify_item(
    image: np.ndarray,
    threshold: float = CONFIDENCE_THRESHOLD,
) -> Screening:
    """Classify one photo and flag low-confidence predictions.

    :param image: Grayscale image with shape ``(28, 28, 1)``.
    :type image: np.ndarray
    :param threshold: Minimum confidence to catalog automatically.
    :type threshold: float
    :return: Predicted category, confidence and review flag.
    :rtype: Screening
    """
    scores = classifier.predict(image[np.newaxis, ...], verbose=0)[0]
    index = int(np.argmax(scores))
    confidence = float(scores[index])

    return Screening(
        label=CLASSES[index],
        confidence=confidence,
        needs_review=confidence < threshold,
    )

In [ ]:
samples = [0, 1, 2, int(misclassified[0])]

plt.figure(figsize=(11, 3))

for position, index in enumerate(samples):
    screening = classify_item(image=x_test[index])
    status = "revisão" if screening.needs_review else "automático"

    plt.subplot(1, len(samples), position + 1)
    plt.imshow(x_test[index].squeeze(), cmap="gray")
    plt.title(
        f"Real: {CLASSES[y_test[index]]}\n"
        f"{screening.label} ({screening.confidence:.0%})\n"
        f"-> {status}",
        fontsize=9,
    )
    plt.axis("off")

plt.tight_layout()
plt.show()